# 01 — Route Features

Steps 1–3: build the route index, develop/QA the per-route feature function on one game, then run all 122 games and concatenate to `data/route_features.parquet`.

Release is measured **relative to each player's own snap depth** (RBs line up ~5 yd behind the LOS, so an absolute LOS+2 threshold is meaningless for them).

In [1]:
import sys, datetime, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")
sys.path.insert(0, "src")
import config
import route_index as ri
import route_features as rf

def _ts():
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

QA_LOG = config.OUTPUT_DIR / "qa_log.txt"
def qa(step, check, value, ok):
    line = f"{_ts()} | {step} | {check} | {value} | {'PASS' if ok else 'FAIL'}"
    with open(QA_LOG, "a") as f:
        f.write(line + "\n")
    print(line)
    return ok

print("config ROOT:", config.ROOT.name)

config ROOT: NYUDataBowl


## STEP 1 — Route index

In [2]:
pff, players, plays = ri.load_sources()
route_index = ri.build_route_index(pff, players)
coverage_index = ri.build_coverage_index(pff)

counts = route_index["group"].value_counts().to_dict()
print("group counts:", counts)
expected = {"chip": 1368, "leak": 362, "clean": 14590}
checks = []
for g, exp in expected.items():
    got = int(counts.get(g, 0))
    checks.append(qa("STEP1", f"count_{g}", f"{got} (exp {exp})", got == exp))
uniq = not route_index.duplicated(["gameId","playId","nflId"]).any()
checks.append(qa("STEP1", "route_index_key_unique", uniq, uniq))
sum_ok = sum(counts.values()) == len(route_index)
checks.append(qa("STEP1", "group_sum_eq_len", f"{sum(counts.values())}=={len(route_index)}", sum_ok))
trk_games = {int(p.stem.split("_")[1]) for p in config.TRACKING_DIR.glob("tracking_*.csv")}
missing = set(route_index["gameId"]) - trk_games
checks.append(qa("STEP1", "all_games_have_tracking", f"missing={len(missing)}", len(missing)==0))
print("\ncounts by group x officialPosition:")
print(pd.crosstab(route_index["group"], route_index["officialPosition"]))
step1_pass = all(checks)
print(f"\nSTEP 1: ROUTE INDEX | {'PASS' if step1_pass else 'FAIL'} | chip={counts.get('chip')} leak={counts.get('leak')} clean={counts.get('clean')}")
assert step1_pass, "STEP 1 STOP"


group counts: {'clean': 14590, 'chip': 1368, 'leak': 362}
2026-11-10 12:40:40 | STEP1 | count_chip | 1368 (exp 1368) | PASS
2026-11-10 12:40:40 | STEP1 | count_leak | 362 (exp 362) | PASS
2026-11-10 12:40:40 | STEP1 | count_clean | 14590 (exp 14590) | PASS
2026-11-10 12:40:40 | STEP1 | route_index_key_unique | True | PASS
2026-11-10 12:40:40 | STEP1 | group_sum_eq_len | 16320==16320 | PASS
2026-11-10 12:40:40 | STEP1 | all_games_have_tracking | missing=0 | PASS

counts by group x officialPosition:
officialPosition   FB    RB    TE
group                            
chip               55   541   772
clean             304  6395  7891
leak               15    18   329

STEP 1: ROUTE INDEX | PASS | chip=1368 leak=362 clean=14590


## STEP 2 — Feature function, tested on one game

In [3]:
def read_game_tracking(game_id, cols=None):
    path = config.TRACKING_DIR / f"tracking_{game_id}.csv"
    usecols = cols or ["gameId","playId","nflId","frameId","event","x","y","playDirection"]
    return pd.read_csv(path, usecols=usecols)

chip_by_game = route_index[route_index["group"]=="chip"].groupby("gameId").size().sort_index()
test_game = int(chip_by_game[chip_by_game >= 3].index[0])
print("test game:", test_game, "chip routes:", int(chip_by_game.loc[test_game]))
trk = read_game_tracking(test_game)
routes_g = route_index[route_index["gameId"]==test_game]
cov_g = coverage_index[coverage_index["gameId"]==test_game]
feat, dropped = rf.compute_route_features(trk, routes_g, cov_g, plays)
print("features:", len(feat), "dropped plays:", len(dropped))
print("columns match spec:", list(feat.columns) == rf.FEATURE_COLUMNS)
feat.head()


test game: 2021090900 chip routes: 8


features: 188 dropped plays: 0
columns match spec: True


,gameId,playId,nflId,group,officialPosition,displayName,t_throw,t_release,depth_at_throw,sep_at_throw,end_event,los_check_diff_yd,depth_at_snap,depth_gain_at_throw,t_release_rel,released_rel,t_release_rel_capped
0,2021090900,97,35481,clean,TE,Rob Gronkowski,3.4,1.4,5.32,1.914001,pass_forward,1.44,0.09,5.23,1.5,True,1.5
1,2021090900,97,39985,clean,RB,Giovani Bernard,3.4,NaN,-3.87,9.825808,pass_forward,1.44,-4.01,0.14,NaN,False,3.4
2,2021090900,137,43293,clean,RB,Ezekiel Elliott,2.6,1.6,9.05,4.580753,pass_forward,0.18,-2.30,11.35,1.2,True,1.2
3,2021090900,137,45532,clean,TE,Blake Jarwin,2.6,1.9,2.97,2.504596,pass_forward,0.18,-2.25,5.22,1.1,True,1.1
4,2021090900,137,46206,clean,TE,Dalton Schultz,2.6,1.3,3.93,6.048347,pass_forward,0.18,-1.38,5.31,1.0,True,1.0


In [4]:
# Plot depth vs time for 3 chip routes, with snap-relative release line
chip_rows = feat[feat["group"]=="chip"].head(3)
trk_n = rf._normalize_coords(trk)
fig, axes = plt.subplots(1, len(chip_rows), figsize=(5*len(chip_rows), 4), squeeze=False)
for ax, (_, cr) in zip(axes[0], chip_rows.iterrows()):
    g,p,n = cr["gameId"], cr["playId"], cr["nflId"]
    pt = trk_n[(trk_n.gameId==g)&(trk_n.playId==p)]
    snap = pt[pt.event.isin(config.SNAP_EVENTS)]["frameId"].min()
    end = pt[(pt.event.isin(config.END_EVENTS))&(pt.frameId>snap)]["frameId"].min()
    los = float(pt[(pt.frameId==snap)&(pt.nflId.isna())]["x"].iloc[0])
    pl = pt[(pt.nflId==n)&(pt.frameId>=snap)&(pt.frameId<=end)].sort_values("frameId")
    t = (pl["frameId"].to_numpy()-snap)/config.FPS
    depth = pl["x"].to_numpy()-los
    ax.plot(t, depth, marker=".")
    ax.axvline(0, color="g", ls="--", label="snap")
    ax.axvline((end-snap)/config.FPS, color="r", ls="--", label="end")
    ax.axhline(config.RELEASE_BUFFER_YD, color="gray", ls=":", label="LOS+2 (abs)")
    ax.axhline(cr["depth_at_snap"]+config.RELEASE_BUFFER_YD, color="purple", ls=":", label="snapdepth+2 (rel)")
    ax.set_title(f"{cr['displayName']}\n{cr['officialPosition']} chip")
    ax.set_xlabel("time since snap (s)"); ax.set_ylabel("depth past LOS (yd)")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(config.OUTPUT_DIR / "qa_chip_traces.png", dpi=90)
print("saved outputs/qa_chip_traces.png"); plt.show()


saved outputs/qa_chip_traces.png


/var/folders/wv/v_v9ssrx0mz9q14xzm9_hq140000gn/T/ipykernel_55712/2281291003.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved outputs/qa_chip_traces.png"); plt.show()


In [5]:
# STEP 2 checks
checks = []
# LOS sanity is a DATASET-level property (gated in STEP 3 on all plays).
# Here we log the single-game number as informational only: one 92-play
# game is too small to hold to a 95% tail bar.
_los_per_play = feat.drop_duplicates(["gameId","playId"])["los_check_diff_yd"]
los_mean = _los_per_play.mean(); los_u15 = (_los_per_play < 1.5).mean()
qa("STEP2", "los_one_game_info_mean_u1.5", f"mean={los_mean:.3f} u1.5={los_u15:.3f}", los_mean < 1.0)
tt = feat["t_throw"]; tt_ok = ((tt>=1.0)&(tt<=6.0)).mean()
checks.append(qa("STEP2", "t_throw_1to6s_frac", f"{tt_ok:.3f}", tt_ok >= 0.95))
sep = feat["sep_at_throw"].dropna(); sep_nonneg = bool((sep>=0).all())
checks.append(qa("STEP2", "sep_nonneg", str(sep_nonneg), sep_nonneg))
qa("STEP2", "sep_over_20yd_flag", int((sep>20).sum()), True)
rel = feat.dropna(subset=["t_release"])
checks.append(qa("STEP2", "t_release_le_t_throw", str(bool((rel["t_release"]<=rel["t_throw"]).all())), bool((rel["t_release"]<=rel["t_throw"]).all())))
# capped rel release always <= t_throw
cap_ok = bool((feat["t_release_rel_capped"] <= feat["t_throw"] + 1e-9).all())
checks.append(qa("STEP2", "t_release_rel_capped_le_t_throw", str(cap_ok), cap_ok))
# no early-depth explosion
explode=False
for _,cr in feat[feat.group=="chip"].iterrows():
    pt = trk_n[(trk_n.gameId==cr.gameId)&(trk_n.playId==cr.playId)]
    snap = pt[pt.event.isin(config.SNAP_EVENTS)]["frameId"].min()
    los = float(pt[(pt.frameId==snap)&(pt.nflId.isna())]["x"].iloc[0])
    pl = pt[(pt.nflId==cr.nflId)&(pt.frameId>=snap)&(pt.frameId<=snap+5)]
    if ((pl["x"]-los)>=10).any(): explode=True
checks.append(qa("STEP2", "no_early_depth_explosion", f"explode={explode}", not explode))
drop_rate = len(dropped)/max(1, routes_g[["gameId","playId"]].drop_duplicates().shape[0])
checks.append(qa("STEP2", "dropped_play_rate", f"{drop_rate:.3f}", drop_rate<0.05))
print("\nstep2 checks pass (pre manual/RB verify):", all(checks))


2026-11-10 12:40:40 | STEP2 | los_one_game_info_mean_u1.5 | mean=0.613 u1.5=0.947 | PASS
2026-11-10 12:40:40 | STEP2 | t_throw_1to6s_frac | 0.979 | PASS
2026-11-10 12:40:40 | STEP2 | sep_nonneg | True | PASS
2026-11-10 12:40:40 | STEP2 | sep_over_20yd_flag | 0 | PASS
2026-11-10 12:40:40 | STEP2 | t_release_le_t_throw | True | PASS
2026-11-10 12:40:40 | STEP2 | t_release_rel_capped_le_t_throw | True | PASS
2026-11-10 12:40:40 | STEP2 | no_early_depth_explosion | explode=False | PASS
2026-11-10 12:40:40 | STEP2 | dropped_play_rate | 0.000 | PASS

step2 checks pass (pre manual/RB verify): True


In [6]:
# NaN rate: t_release (abs) vs t_release_rel, by group x officialPosition
def nan_rate(df, col): return df[col].isna().mean()
grp = feat.groupby(["group","officialPosition"])
tab = grp.apply(lambda d: pd.Series({
    "n": len(d),
    "nan_t_release_abs": nan_rate(d,"t_release"),
    "nan_t_release_rel": nan_rate(d,"t_release_rel"),
}), include_groups=False)
print(tab.to_string())
# RB NaN rate must drop a lot vs absolute
rb = feat[feat.officialPosition.isin(config.RB_POSITIONS)]
rb_abs = nan_rate(rb,"t_release"); rb_rel = nan_rate(rb,"t_release_rel")
print(f"\nRB NaN: abs={rb_abs:.3f} rel={rb_rel:.3f}")
qa("STEP2","rb_nan_rel_lt_abs", f"abs={rb_abs:.3f} rel={rb_rel:.3f}", rb_rel < rb_abs)
# Bernard-type RB chip should now have a real release near 1.5-1.7s
# Bernard is kept only as a printed example (his lone chip is a genuine
# non-release checkdown, so NaN is correct). The real gate is group-level:
# RB t_release_rel NaN rate must be far below the absolute version, and at
# least one RB chip must produce a real (non-NaN) relative release.
bern = feat[(feat.displayName.str.contains("Bernard"))&(feat.group=="chip")]
if len(bern):
    print(f"Bernard chip t_release_rel (example only) = {float(bern['t_release_rel'].iloc[0])}")
rbchip = feat[(feat.officialPosition.isin(config.RB_POSITIONS))&(feat.group=="chip")]
rbchip_nonnan = int(rbchip["t_release_rel"].notna().sum())
print(f"RB chips in game: {len(rbchip)} | with non-NaN t_release_rel: {rbchip_nonnan}")
qa("STEP2","rb_chip_release_nonnan_exists", f"{rbchip_nonnan}/{len(rbchip)}", rbchip_nonnan>0)


                           n  nan_t_release_abs  nan_t_release_rel
group officialPosition                                            
chip  RB                 4.0           1.000000           0.250000
      TE                 4.0           0.500000           0.250000
clean RB                78.0           0.679487           0.256410
      TE                96.0           0.031250           0.020833
leak  TE                 6.0           0.666667           0.666667

RB NaN: abs=0.695 rel=0.256
2026-11-10 12:40:40 | STEP2 | rb_nan_rel_lt_abs | abs=0.695 rel=0.256 | PASS
Bernard chip t_release_rel (example only) = nan
RB chips in game: 4 | with non-NaN t_release_rel: 3
2026-11-10 12:40:40 | STEP2 | rb_chip_release_nonnan_exists | 3/4 | PASS


True

In [7]:
# MANUAL VERIFY sep_at_throw for one route, by hand (NaN-safe nflId handling)
mv = feat[feat["sep_at_throw"].notna()].iloc[0]
g,p,n = mv["gameId"], mv["playId"], mv["nflId"]
ptn = trk_n[(trk_n.gameId==g)&(trk_n.playId==p)]
snap = ptn[ptn.event.isin(config.SNAP_EVENTS)]["frameId"].min()
end = ptn[(ptn.event.isin(config.END_EVENTS))&(ptn.frameId>snap)]["frameId"].min()
me = ptn[(ptn.nflId==n)&(ptn.frameId==end)][["x","y"]].iloc[0]
cov_ids = set(coverage_index[(coverage_index.gameId==g)&(coverage_index.playId==p)]["nflId"].dropna().astype("int64"))
_end_pl = ptn[(ptn.frameId==end)&(ptn.nflId.notna())]
cov_rows = _end_pl[_end_pl.nflId.astype('int64').isin(cov_ids)][["nflId","x","y"]]
print(f"route player {int(n)} at end frame {int(end)}: x={me.x:.2f} y={me.y:.2f}")
print(cov_rows.to_string(index=False))
dists = np.sqrt((cov_rows["x"]-me.x)**2 + (cov_rows["y"]-me.y)**2)
hand=float(dists.min()); func=float(mv["sep_at_throw"])
print(f"hand={hand:.4f} func={func:.4f} match<=0.01: {abs(hand-func)<=0.01}")
mv_ok = abs(hand-func)<=0.01
qa("STEP2","manual_verify_sep", f"hand={hand:.4f} func={func:.4f}", mv_ok)
print(f"\nSTEP 2: FEATURE FUNCTION | {'PASS' if mv_ok else 'FAIL'} | game={test_game} los_mean={los_mean:.2f} drops={drop_rate:.2%} RB nan abs->rel {rb_abs:.2f}->{rb_rel:.2f}")
assert mv_ok, "STEP 2 STOP"


route player 35481 at end frame 40: x=46.88 y=44.67
  nflId     x     y
43306.0 48.03 43.14
43478.0 55.34 42.82
44904.0 54.39 23.31
44962.0 69.32 30.43
47996.0 46.70 19.09
52459.0 50.26 26.02
hand=1.9140 func=1.9140 match<=0.01: True
2026-11-10 12:40:40 | STEP2 | manual_verify_sep | hand=1.9140 func=1.9140 | PASS

STEP 2: FEATURE FUNCTION | PASS | game=2021090900 los_mean=0.61 drops=0.00% RB nan abs->rel 0.70->0.26


## STEP 3 — Full run with checkpoints

In [8]:
from tqdm import tqdm
# FORCE overwrite of existing part files for this run (pass_forward fallback
# fix): ignore the skip-if-exists check so every game is recomputed.
FORCE = True
games = sorted({int(p.stem.split("_")[1]) for p in config.TRACKING_DIR.glob("tracking_*.csv")})
per_game=[]; all_dropped=[]
for gid in tqdm(games, desc="games"):
    part_path = config.PARTS_DIR / f"route_features_{gid}.parquet"
    routes_g = route_index[route_index["gameId"]==gid]
    if part_path.exists() and not FORCE:
        f = pd.read_parquet(part_path); per_game.append((gid,len(routes_g),len(f),0)); continue
    trk = read_game_tracking(gid)
    cov_g = coverage_index[coverage_index["gameId"]==gid]
    f, dr = rf.compute_route_features(trk, routes_g, cov_g, plays)
    f.to_parquet(part_path, index=False)
    per_game.append((gid,len(routes_g),len(f),len(dr)))
    if len(dr): dr=dr.copy(); dr["gameId_src"]=gid; all_dropped.append(dr)
pg = pd.DataFrame(per_game, columns=["gameId","routes_expected","routes_produced","plays_dropped"])
dropped_all = pd.concat(all_dropped, ignore_index=True) if all_dropped else pd.DataFrame(columns=["gameId","playId","reason"])
print("games:", len(pg), "routes produced:", pg.routes_produced.sum())


games:   0%|          | 0/122 [00:00<?, ?it/s]

games:   1%|          | 1/122 [00:00<00:55,  2.18it/s]

games:   2%|▏         | 2/122 [00:00<00:46,  2.59it/s]

games:   2%|▏         | 3/122 [00:01<00:40,  2.91it/s]

games:   3%|▎         | 4/122 [00:01<00:40,  2.91it/s]

games:   4%|▍         | 5/122 [00:01<00:38,  3.03it/s]

... (115 outputs left out)

games:  99%|█████████▉| 121/122 [00:34<00:00,  3.43it/s]

games: 100%|██████████| 122/122 [00:35<00:00,  3.45it/s]

games: 100%|██████████| 122/122 [00:35<00:00,  3.47it/s]

games: 122 routes produced: 16272


In [9]:
parts = sorted(config.PARTS_DIR.glob("route_features_*.parquet"))
full = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
full.to_parquet(config.DATA_DIR / "route_features.parquet", index=False)
print("route_features.parquet rows:", len(full))
checks=[]
checks.append(qa("STEP3","part_files_count", len(parts), len(parts)==122))
prod = full["group"].value_counts().to_dict(); exp = route_index["group"].value_counts().to_dict()
for g in ["chip","leak","clean"]:
    cov = prod.get(g,0)/exp[g]; checks.append(qa("STEP3",f"coverage_{g}", f"{cov:.3f}", cov>=0.95))
total_plays = route_index[["gameId","playId"]].drop_duplicates().shape[0]
drop_rate = len(dropped_all)/max(1,total_plays)
checks.append(qa("STEP3","total_dropped_rate", f"{drop_rate:.4f}", drop_rate<0.05))
if len(dropped_all):
    print("dropped reasons:", dropped_all["reason"].str.replace(r"_\d+_","_<id>_",regex=True).value_counts().to_dict())
cols_ok = list(full.columns)==rf.FEATURE_COLUMNS
checks.append(qa("STEP3","columns_match_spec", str(cols_ok), cols_ok))
# full-dataset LOS sanity gate: mean < 1.0 yd AND 95% of plays under 1.5 yd
_losf = full.drop_duplicates(["gameId","playId"])["los_check_diff_yd"].dropna()
los_full_mean = _losf.mean(); los_full_u15 = (_losf < 1.5).mean()
los_full_ok = (los_full_mean < 1.0) and (los_full_u15 >= 0.95)
checks.append(qa("STEP3","los_mean_lt1_and_95pct_under_1.5", f"mean={los_full_mean:.3f} u1.5={los_full_u15:.3f}", los_full_ok))
print("dtypes:\n", full.dtypes)
fake = config.DATA_DIR / "route_features_fake.parquet"
if fake.exists():
    same = list(pd.read_parquet(fake).columns)==list(full.columns)
    qa("STEP3","fake_columns_match", str(same), same)
else:
    qa("STEP3","fake_columns_match","fake_absent", True)
print("\nNaN rate per column:"); print(full.isna().mean().round(4).to_string())
tr_chip=full[full.group=="chip"]["t_release"].isna().mean(); tr_clean=full[full.group=="clean"]["t_release"].isna().mean()
checks.append(qa("STEP3","t_release_nan_chip_gt_clean", f"chip={tr_chip:.3f} clean={tr_clean:.3f}", tr_chip>tr_clean))
print("\nmedian by group:")
print(full.groupby("group")[["t_throw","t_release","t_release_rel_capped","depth_at_throw","depth_gain_at_throw","sep_at_throw"]].median())
step3_pass = all(checks)
print(f"\nSTEP 3: FULL RUN | {'PASS' if step3_pass else 'FAIL'} | parts={len(parts)} rows={len(full)} drop={drop_rate:.2%}")
assert all(prod.get(g,0)/exp[g] >= 0.90 for g in exp), "STEP 3 STOP: coverage < 90%"


route_features.parquet rows: 16272
2026-11-10 12:41:16 | STEP3 | part_files_count | 122 | PASS
2026-11-10 12:41:16 | STEP3 | coverage_chip | 0.995 | PASS
2026-11-10 12:41:16 | STEP3 | coverage_leak | 1.000 | PASS
2026-11-10 12:41:16 | STEP3 | coverage_clean | 0.997 | PASS
2026-11-10 12:41:16 | STEP3 | total_dropped_rate | 0.0029 | PASS
dropped reasons: {'no_snap_event': 24}
2026-11-10 12:41:16 | STEP3 | columns_match_spec | True | PASS
2026-11-10 12:41:16 | STEP3 | los_mean_lt1_and_95pct_under_1.5 | mean=0.458 u1.5=0.968 | PASS
dtypes:
 gameId                    int64
playId                    int64
nflId                     int64
group                       str
officialPosition            str
displayName                 str
t_throw                 float64
t_release               float64
depth_at_throw          float64
sep_at_throw            float64
end_event                   str
los_check_diff_yd       float64
depth_at_snap           float64
depth_gain_at_throw     float64
t_release

In [10]:
# repeat Step 2 sanity on a SECOND game
game2 = int(chip_by_game[chip_by_game>=3].index[1])
trk2 = read_game_tracking(game2); r2=route_index[route_index.gameId==game2]; c2=coverage_index[coverage_index.gameId==game2]
f2,d2 = rf.compute_route_features(trk2,r2,c2,plays)
_lp2=f2.drop_duplicates(["gameId","playId"])["los_check_diff_yd"]
los2_mean=_lp2.mean(); los2_u15=(_lp2<1.5).mean(); los2=(los2_mean<1.0) and (los2_u15>=0.95)
tt2=((f2["t_throw"]>=1.0)&(f2["t_throw"]<=6.0)).mean()
dr2=len(d2)/max(1,r2[["gameId","playId"]].drop_duplicates().shape[0])
cap2=bool((f2["t_release_rel_capped"]<=f2["t_throw"]+1e-9).all())
qa("STEP3",f"game2_{game2}_los_info", f"mean={los2_mean:.3f} u1.5={los2_u15:.3f}", los2_mean<1.0)
qa("STEP3",f"game2_{game2}_t_throw_ok", f"{tt2:.3f}", tt2>=0.95)
qa("STEP3",f"game2_{game2}_drop_rate", f"{dr2:.3f}", dr2<0.05)
qa("STEP3",f"game2_{game2}_cap_le_throw", str(cap2), cap2)
print(f"second game {game2}: los_mean={los2_mean:.2f} t_throw_ok={tt2:.2f} drops={dr2:.2%} cap_ok={cap2}")


2026-11-10 12:41:16 | STEP3 | game2_2021091200_los_info | mean=0.402 u1.5=0.972 | PASS
2026-11-10 12:41:16 | STEP3 | game2_2021091200_t_throw_ok | 0.989 | PASS
2026-11-10 12:41:16 | STEP3 | game2_2021091200_drop_rate | 0.014 | PASS
2026-11-10 12:41:16 | STEP3 | game2_2021091200_cap_le_throw | True | PASS
second game 2021091200: los_mean=0.40 t_throw_ok=0.99 drops=1.37% cap_ok=True


## STEP 4 — Handoff summary for Isaac

In [11]:
full = pd.read_parquet(config.DATA_DIR / "route_features.parquet")
size_mb = (config.DATA_DIR / "route_features.parquet").stat().st_size/1e6
msg=["@Isaac — route_features.parquet is ready.",
 f"rows: {len(full)} | size: {size_mb:.2f} MB",
 "rows per group: "+str(full['group'].value_counts().to_dict()),
 "position split: "+str(full.groupby(['group','officialPosition']).size().to_dict()),
 "NaN rates: "+str(full.isna().mean().round(3).to_dict()),
 "columns (order): "+", ".join(full.columns),
 "KEY: use t_release_rel_capped and depth_gain_at_throw (snap-relative) for RB-inclusive analysis; "
 "t_release/depth_at_throw are LOS-absolute and NaN-heavy for RBs. end_event in {qb_sack,qb_strip_sack,run} = no throw, drop in analysis. "
 "Never pool TE and RB. Coords normalized to face right."]
handoff="\n".join(msg); print(handoff)
with open(config.OUTPUT_DIR/"handoff_to_isaac.txt","w") as fh: fh.write(handoff+"\n")
print("\n5 sample rows:"); print(full.sample(5, random_state=config.RANDOM_SEED).to_string(index=False))
if size_mb>50:
    print(f"\n*** parquet {size_mb:.1f} MB (>50MB): do NOT commit. Regenerate: python3 -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=dualduty notebooks/01_route_features.ipynb ***")
else:
    print(f"\nReminder: git add data/route_features.parquet && git commit -m 'route features' && git push  ({size_mb:.1f} MB ok)")
qa("STEP4","handoff_written", f"{len(full)} rows {size_mb:.2f}MB", True)
print("\nSTEP 4: HANDOFF | PASS")


@Isaac — route_features.parquet is ready.
rows: 16272 | size: 0.48 MB
rows per group: {'clean': 14549, 'chip': 1361, 'leak': 362}
position split: {('chip', 'FB'): 55, ('chip', 'RB'): 539, ('chip', 'TE'): 767, ('clean', 'FB'): 304, ('clean', 'RB'): 6378, ('clean', 'TE'): 7867, ('leak', 'FB'): 15, ('leak', 'RB'): 18, ('leak', 'TE'): 329}
NaN rates: {'gameId': 0.0, 'playId': 0.0, 'nflId': 0.0, 'group': 0.0, 'officialPosition': 0.0, 'displayName': 0.0, 't_throw': 0.0, 't_release': 0.389, 'depth_at_throw': 0.0, 'sep_at_throw': 0.0, 'end_event': 0.0, 'los_check_diff_yd': 0.0, 'depth_at_snap': 0.0, 'depth_gain_at_throw': 0.0, 't_release_rel': 0.147, 'released_rel': 0.0, 't_release_rel_capped': 0.0}
columns (order): gameId, playId, nflId, group, officialPosition, displayName, t_throw, t_release, depth_at_throw, sep_at_throw, end_event, los_check_diff_yd, depth_at_snap, depth_gain_at_throw, t_release_rel, released_rel, t_release_rel_capped
KEY: use t_release_rel_capped and depth_gain_at_throw (